# DataFog PII-NER v1.4 — Training with Expanded Entity Coverage

This notebook runs on any GPU instance (Lambda H100, Colab A100/T4, etc.). It:
1. Sets up the environment
2. Runs the entity audit to verify all 41 types have training data
3. Trains the model

**Recommended:** Lambda H100 (80GB) for ~2-3 hour training

## 1. Setup

In [ ]:
# Clone the repo
!git clone https://github.com/DataFog/datafog-labs.git
%cd datafog-labs/pii-ner-v1

In [ ]:
# Install dependencies
!pip install -e ".[dev]" -q
!pip install wandb -q

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")

In [ ]:
# Run tests to verify everything works
!python -m pytest tests/ -v --tb=short

## 2. Entity Audit

Verify all 41 entity types have non-zero training examples.
This downloads 7 datasets from HuggingFace + loads local synthetic data.

In [ ]:
# Download all datasets first
!python scripts/download_data.py

In [ ]:
# Run entity audit — this is the acceptance gate
!python scripts/entity_audit.py

## 3. Training Config

Auto-detects GPU and sets batch size/epochs accordingly.
H100 80GB: batch 32, 7 epochs (~2-3 hours).
A100 40GB: batch 16, 7 epochs (~5-6 hours).
T4 16GB: batch 4, 7 epochs (~10-12 hours).

In [ ]:
import yaml
import torch

# Auto-detect GPU and set batch size accordingly
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

if vram_gb >= 70:  # H100 80GB
    batch_size = 32
    grad_accum = 1  # effective = 32
    workers = 4
elif vram_gb >= 40:  # A100 40GB
    batch_size = 16
    grad_accum = 2  # effective = 32
    workers = 4
elif vram_gb >= 15:  # T4
    batch_size = 4
    grad_accum = 8  # effective = 32
    workers = 2
else:
    batch_size = 2
    grad_accum = 16  # effective = 32
    workers = 0

# 7 epochs: 3 full backbone + 4 head-only (v1.3 showed best by epoch 5-6)
epochs = 7

print(f"GPU: {gpu_name} ({vram_gb:.0f} GB)")
print(f"Batch: {batch_size} x {grad_accum} = {batch_size * grad_accum} effective")
print(f"Epochs: {epochs}")

config = {
    "model": {
        "backbone": "microsoft/deberta-v3-xsmall",
        "char_embed_dim": 50,
        "char_vocab_size": 256,
        "char_cnn_filters": [50, 50, 50],
        "char_cnn_widths": [3, 4, 5],
        "max_char_len": 20,
        "dropout": 0.1,
    },
    "data": {
        "max_seq_len": 256,
        "val_ratio": 0.1,
        "test_ratio": 0.1,
        "seed": 42,
        "oversample_tiers": [1],
        "oversample_factor": 3,
    },
    "training": {
        "epochs": epochs,
        "batch_size": batch_size,
        "gradient_accumulation_steps": grad_accum,
        "lr_backbone": 1.0e-5,
        "lr_head": 1.0e-3,
        "lr_scheduler_type": "cosine",
        "warmup_steps": 500,
        "weight_decay": 0.01,
        "eval_strategy": "epoch",
        "save_strategy": "epoch",
        "metric_for_best_model": "overall_f1",
        "logging_steps": 50,
        "dataloader_num_workers": workers,
        "save_total_limit": 3,
        "output_dir": "runs/v1.4",
        "run_name": f"pii-ner-v1.4-{gpu_name.split()[0].lower()}",
        "freeze_backbone_after_epoch": 3,
        "tier_weights": {1: 3.0, 2: 2.0, 3: 1.5, 4: 1.0},
        "tier_weights_after_epoch_2": {1: 2.0, 2: 1.5, 3: 1.25, 4: 1.0},
    },
    "wandb": {
        "enabled": False,  # Set to True and run wandb.login() to enable
        "project": "datafog-pii-ner",
    },
}

with open("configs/v1.4.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"\nConfig written to configs/v1.4.yaml")

## 4. (Optional) WandB Login

Uncomment and run if you want experiment tracking.

In [ ]:
# import wandb
# wandb.login()
# # Then set config["wandb"]["enabled"] = True above and re-run that cell

## 5. Train

In [ ]:
!python scripts/train_v1.3.py --config configs/v1.4.yaml

## 6. Results

In [ ]:
import json
from pathlib import Path

results_path = Path("runs/v1.4/best_model/test_results.json")
if results_path.exists():
    results = json.loads(results_path.read_text())
    print(f"Overall F1:        {results.get('eval_overall_f1', 0):.4f}")
    print(f"Overall Precision: {results.get('eval_overall_precision', 0):.4f}")
    print(f"Overall Recall:    {results.get('eval_overall_recall', 0):.4f}")
    print()
    for tier in [1, 2, 3, 4]:
        key = f"eval_tier_{tier}_recall"
        target = {1: 0.98, 2: 0.95, 3: 0.90, 4: 0.85}[tier]
        actual = results.get(key, 0)
        status = "PASS" if actual >= target else "FAIL"
        print(f"Tier {tier} recall: {actual:.4f}  (target >= {target})  [{status}]")
    print()
    # Per-type F1
    type_f1 = {k.replace('eval_type_', '').replace('_f1', ''): v 
               for k, v in results.items() if k.startswith('eval_type_') and k.endswith('_f1')}
    print("Per-entity F1:")
    for name, f1 in sorted(type_f1.items(), key=lambda x: x[1], reverse=True):
        print(f"  {name:30s} {f1:.4f}")
else:
    print("No results yet — training may still be running.")

## 7. Save Model to Google Drive (Optional)

Persist the trained model across Colab sessions.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r runs/v1.4-colab/best_model /content/drive/MyDrive/datafog-pii-ner-v1.4/